In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn import pipeline
from sklearn.compose import ColumnTransformer,  make_column_selector
from imblearn.pipeline import Pipeline
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv('adult.csv')

In [3]:
df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [4]:
df.shape

(48842, 15)

In [5]:
df.duplicated().sum()

np.int64(52)

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.isna().sum()

age                0
workclass          0
fnlwgt             0
education          0
educational-num    0
marital-status     0
occupation         0
relationship       0
race               0
gender             0
capital-gain       0
capital-loss       0
hours-per-week     0
native-country     0
income             0
dtype: int64

In [9]:
df.drop(columns=['fnlwgt', 'educational-num','race'], inplace=True)

In [10]:
df.replace('?', np.nan, inplace=True)

In [11]:
df.dropna(inplace=True)

In [12]:
df.shape

(45175, 12)

In [13]:
cat_cols = df.select_dtypes(include=['object']).columns
df[cat_cols] = df[cat_cols].astype(str)

In [14]:
df.shape

(45175, 12)

In [15]:
df['income'] = df['income'].astype(str)

In [16]:
X = df.drop('income', axis=1)
y = df.income

In [17]:
df.shape

(45175, 12)

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [19]:
cat_cols = df.select_dtypes('object')
num_cols = df.select_dtypes('number')

In [20]:
cat_cols

,workclass,education,marital-status,occupation,relationship,gender,native-country,income
0,Private,11th,Never-married,Machine-op-inspct,Own-child,Male,United-States,<=50K
1,Private,HS-grad,Married-civ-spouse,Farming-fishing,Husband,Male,United-States,<=50K
2,Local-gov,Assoc-acdm,Married-civ-spouse,Protective-serv,Husband,Male,United-States,>50K
3,Private,Some-college,Married-civ-spouse,Machine-op-inspct,Husband,Male,United-States,>50K
5,Private,10th,Never-married,Other-service,Not-in-family,Male,United-States,<=50K
...,...,...,...,...,...,...,...,...
48837,Private,Assoc-acdm,Married-civ-spouse,Tech-support,Wife,Female,United-States,<=50K
48838,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,Male,United-States,>50K
48839,Private,HS-grad,Widowed,Adm-clerical,Unmarried,Female,United-States,<=50K
48840,Private,HS-grad,Never-married,Adm-clerical,Own-child,Male,United-States,<=50K


# LOGISTIC REGRESSION

In [21]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [22]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [23]:
preprocessor = ColumnTransformer([
                            ("num", num_pipeline, make_column_selector(dtype_include=np.number)),
                            ("cat", cat_pipeline, make_column_selector(dtype_include=object))
])

In [26]:
model_pipeline_1 = Pipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(sampling_strategy='minority')),
    ("model", LogisticRegression())    
])
model_pipeline_1.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [27]:
y_pred = model_pipeline_1.predict(X_test)

In [28]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       <=50K       0.94      0.80      0.86      6842
        >50K       0.57      0.84      0.68      2193

    accuracy                           0.81      9035
   macro avg       0.76      0.82      0.77      9035
weighted avg       0.85      0.81      0.82      9035



# KNN

In [29]:
model_pipeline_2 = Pipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(sampling_strategy='minority')),
    ("model", KNeighborsClassifier())
])
model_pipeline_2.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [30]:
y_pred = model_pipeline_2.predict(X_test)

In [31]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       <=50K       0.91      0.80      0.85      6842
        >50K       0.54      0.76      0.63      2193

    accuracy                           0.79      9035
   macro avg       0.73      0.78      0.74      9035
weighted avg       0.82      0.79      0.80      9035



# SVC

In [32]:
model_pipeline_3 = Pipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(sampling_strategy='minority')),
    ("model", SVC(kernel='rbf', C=1.0))
])
model_pipeline_3.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [33]:
y_pred = model_pipeline_3.predict(X_test)

In [34]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       <=50K       0.94      0.79      0.86      6842
        >50K       0.57      0.85      0.68      2193

    accuracy                           0.81      9035
   macro avg       0.76      0.82      0.77      9035
weighted avg       0.85      0.81      0.82      9035



# DECISION TREE

In [35]:
model_pipeline_4 = Pipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(sampling_strategy='minority')),
    ("model", DecisionTreeClassifier(max_depth=10, random_state=42))
])
model_pipeline_4.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [36]:
y_pred = model_pipeline_4.predict(X_test)

In [37]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       <=50K       0.95      0.75      0.84      6842
        >50K       0.53      0.87      0.66      2193

    accuracy                           0.78      9035
   macro avg       0.74      0.81      0.75      9035
weighted avg       0.85      0.78      0.79      9035



In [38]:
import pickle
with open("svc_model_3.pkl", "wb") as file:
    pickle.dump(model_pipeline_3, file)
print("SVC model saved successfully!")

SVC model saved successfully!
